# 02. 데이터 전처리 및 분할 검증

> README **5.1 데이터 개요**, **5.2 Target 클래스 분포**(그림 `figures/target_distribution.png`), **6.2 데이터 전처리**, **6.3 분할 검증**에 대응하는 노트북입니다.

**목적**
01 EDA에서 내린 결정을 원본 데이터에 적용해 팀 공통 전처리 데이터(`data/processed/Faults_processed.csv`)를 만들고, 팀 공통 Train/Test 분할이 올바른지 검증합니다.

**전제**
- `src/make_split.py`를 먼저 실행해 `data/split/train_idx.csv`, `test_idx.csv`가 있어야 합니다.
- 여기서 하는 전처리는 **행 단위 변환·열 제거**뿐이며, 데이터로부터 통계를 학습하는 처리(Scaling 등)는 없습니다. 따라서 전체 데이터에 적용해도 Test 정보가 새지 않습니다. 통계를 학습하는 처리는 모델링 단계의 Pipeline 안에서 Train fold 기준으로만 합니다.

| 섹션 | 내용 | 근거 |
|---|---|---|
| 1 | 품질 재확인 (결측·완전 중복·유사 중복) | 01 EDA 4장, README 6.2 |
| 2 | Target 단일화 및 표기 수정 | 01 EDA 4장 (누수 방지), README 5.1 |
| 3 | 강종 변수 정리 | 01 EDA 4·6장 |
| 4 | 저장 및 검증 | |
| 5 | Train/Test 분할 검증 | README 6.3 |
| 6 | README 그림: 유형별 건수와 Baseline F1 | README 5.2, 03 Baseline과 같은 설정 (Train만) |
| 7 | 전처리 결정 기록 | README 6.2 |

In [ ]:
import sys
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 노트북 위치와 무관하게 프로젝트 루트(src/config.py가 있는 폴더)를 찾는다
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

# 한글 폰트: OS별로 있는 폰트를 순서대로 사용
plt.rcParams["font.family"] = ["Malgun Gothic", "AppleGothic", "NanumGothic",
                               "Noto Sans CJK KR", "Noto Sans CJK JP", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
pd.set_option("display.width", 200)

## 1. 품질 재확인 (README 6.2)

EDA는 Train만 봤으므로, 전처리를 적용할 **전체 데이터** 기준으로 한 번 더 확인합니다. 특히 **유사 중복**은 한 쌍이 Train과 Test에 나뉘어 있을 수 있어 Train만으로는 찾을 수 없으므로 여기서 확인합니다.

In [ ]:
df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "Faults.csv")
print("원본 크기:", df.shape)
print("결측치 합계:", int(df.isna().sum().sum()))
print("중복 행 수 :", int(df.duplicated().sum()))

**확인 결과:** 전체 데이터에서도 결측치·완전 중복 행이 0건이므로 별도 처리하지 않습니다.

### 유사 중복

완전히 같은 행은 없지만, 측정값에는 작은 오차가 있을 수 있으므로 **조금만 달라도 값이 바뀌는 측정값**(길이 방향 좌표·면적·둘레·밝기)이 모두 같은 행을 찾습니다. 이런 값까지 모두 같다면 같은 결함을 두 번 기록했을 가능성이 있습니다.

In [ ]:
tr = pd.read_csv(PROJECT_ROOT / "data" / "split" / "train_idx.csv")["index"]
te = pd.read_csv(PROJECT_ROOT / "data" / "split" / "test_idx.csv")["index"]
split_of = lambda i: "Train" if i in set(tr) else "Test"

SENSITIVE = ["Y_Minimum", "Y_Maximum", "Pixels_Areas", "X_Perimeter", "Y_Perimeter",
             "Sum_of_Luminosity", "Minimum_of_Luminosity", "Maximum_of_Luminosity"]
suspect = df.index[df.duplicated(SENSITIVE, keep=False)].tolist()
print("민감한 측정값이 모두 같은 행:", suspect, "→", [split_of(i) for i in suspect])

# 비교용: 다른 팀원 분석(dongjin_data_quality.ipynb)에서 판단을 보류한 쌍
held = [529, 678]
print("판단 보류 쌍:", held, "→", [split_of(i) for i in held])

def compare(rows):
    t = df.loc[rows].drop(columns=["Pastry", "Z_Scratch", "K_Scatch", "Stains", "Dirtiness", "Bumps", "Other_Faults"]).T
    t["같음"] = t.nunique(axis=1) == 1
    return t

display(compare(suspect))
compare(held)

**확인 결과**
- **1775 · 1776**: 길이 방향 좌표, 면적, 둘레, 밝기 네 값, 모양 지표 대부분과 유형(Other_Faults)이 같고, `Length_of_Conveyer`(2 차이)와 가로 좌표(6 차이), `Edges_Index`(소수점 셋째 자리)만 조금 다릅니다. 1775는 Test, 1776은 Train에 있습니다.
- **529 · 678**: 가로 시작 좌표와 길이 방향 좌표(8 이내), 유형(K_Scratch)은 같지만 `Length_of_Conveyer`가 42 차이 나고 모양 지표도 어긋납니다. 둘 다 Train에 있습니다.

**판단**
- 1775 · 1776은 밝기처럼 민감한 값까지 모두 같아 같은 결함의 이중 기록으로 **의심**되지만, 강판 번호가 없어 확정할 방법이 없습니다. 같은 자리에 비슷한 결함이 실제로 반복되는 경우도 있으므로(01 EDA 10장) **제거하지 않습니다.** Test 한 건이 사실상 이미 본 결함일 수 있다는 점은 한계로 기록합니다.
- 529 · 678은 같은 결함이라고 보기 더 어려워 **유지합니다.**

## 2. Target 단일화 및 표기 수정

- 원본 열 이름 `K_Scatch`를 `K_Scratch`로 수정합니다. 결과표·대시보드 표기를 통일하기 위함입니다.
- 7개 원-핫 열에서 1인 열의 이름을 `target`으로 만들고, 원-핫 열은 제거합니다. 정답 열이 입력에 남으면 데이터 누수가 됩니다.

In [ ]:
df = df.rename(columns={"K_Scatch": "K_Scratch"})

TARGETS = ["Pastry", "Z_Scratch", "K_Scratch", "Stains",
           "Dirtiness", "Bumps", "Other_Faults"]

# 원-핫 구조 검증: 정확히 하나만 1이어야 idxmax 결과가 유일하다
assert (df[TARGETS].sum(axis=1) == 1).all(), "원-핫 합이 1이 아닌 행이 있습니다."

df["target"] = df[TARGETS].idxmax(axis=1)
df_processed = df.drop(columns=TARGETS)

print("변환 전:", df.shape, "→ 변환 후:", df_processed.shape)
df_processed["target"].value_counts()

In [ ]:
# README 5.1 클래스 비율 (전체 1,941건 기준)
full_cnt = df_processed["target"].value_counts()
full_pct = full_cnt / full_cnt.sum() * 100
print(f"최다 {full_cnt.idxmax()} {full_pct.max():.2f}% / 최소 {full_cnt.idxmin()} {full_pct.min():.2f}% / "
      f"최대÷최소 {full_cnt.max() / full_cnt.min():.2f}배")

README 5.1의 클래스 비율은 **전체 데이터 기준**입니다. 01 EDA의 Train 기준 값(34.66% ~ 2.84%, 12.23배)과는 기준만 다르고, 층화 분할이라 거의 같습니다.

## 3. 강종 변수 정리

EDA에서 `TypeOfSteel_A300`과 `TypeOfSteel_A400`이 Train에서 완전히 상호배타적임을 확인했습니다. 전체 데이터에서도 같은지 확인한 뒤 `A400`을 제거합니다. 남은 `TypeOfSteel_A300`은 1 = A300, 0 = A400을 의미합니다.

In [ ]:
steel = df_processed[["TypeOfSteel_A300", "TypeOfSteel_A400"]]
print(steel.value_counts().sort_index().to_string())
assert (steel.sum(axis=1) == 1).all(), "두 강종 열이 상호배타적이지 않습니다."

df_processed = df_processed.drop(columns=["TypeOfSteel_A400"])
print("\n최종 크기:", df_processed.shape, "(Feature 26개 + target)")

## 4. 저장 및 검증

In [ ]:
out_path = PROJECT_ROOT / "data" / "processed" / "Faults_processed.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df_processed.to_csv(out_path, index=False)

check = pd.read_csv(out_path)
print("저장 경로:", out_path.relative_to(PROJECT_ROOT))
print("크기     :", check.shape)
print("결측치   :", int(check.isna().sum().sum()))
print("중복 행  :", int(check.duplicated().sum()))
print("원-핫/강종 열 잔존:", [c for c in TARGETS + ["TypeOfSteel_A400"] if c in check.columns])

## 5. Train/Test 분할 검증 (README 6.3)

`make_split.py`는 `config.py`의 설정(80:20, `random_state=42`, Target 기준 층화)으로 인덱스를 만듭니다. 여기서는 ① 분할 자체에 문제가 없는지, ② 유형별 비율이 유지되는지, ③ Test에서 결과가 불안정할 유형이 있는지 확인합니다.

In [ ]:
# tr, te는 1장에서 읽음
y_all = df_processed["target"]

checks = {
    "Train·Test 겹치는 행": len(set(tr) & set(te)),
    "누락된 행": len(df_processed) - len(set(tr) | set(te)),
    "Train 내 중복 인덱스": len(tr) - tr.nunique(),
    "Test 내 중복 인덱스": len(te) - te.nunique(),
}
for k, v in checks.items():
    print(f"{'OK' if v == 0 else 'NG'}  {k}: {v}")
print(f"\nTrain {len(tr)}건 / Test {len(te)}건 (Test 비율 {len(te) / len(df_processed):.4f})")

In [ ]:
ratio = pd.DataFrame({
    "Train 건수": y_all.loc[tr].value_counts(),
    "Test 건수": y_all.loc[te].value_counts(),
    "전체 비율": y_all.value_counts(normalize=True),
    "Train 비율": y_all.loc[tr].value_counts(normalize=True),
    "Test 비율": y_all.loc[te].value_counts(normalize=True),
})
ratio["전체와 최대 차이"] = ratio[["Train 비율", "Test 비율"]].sub(ratio["전체 비율"], axis=0).abs().max(axis=1)
display(ratio)
print(f"최대 비율 차이: {ratio['전체와 최대 차이'].max():.4f}")

print("\n[Test 20건 미만 유형 → 1건 차이로 Recall이 크게 움직임]")
for name, n in ratio["Test 건수"][ratio["Test 건수"] < 20].items():
    print(f"  {name:10s} Test {n}건 → 1건당 Recall {1 / n:.3f}")

In [ ]:
# 분할 확인용 그림 (파일로 저장하지 않음. README 그림은 6장에서 저장)
order = y_all.value_counts().sort_values().index
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cnt = y_all.value_counts()[order]
pct = cnt / cnt.sum() * 100
b = axes[0].barh(cnt.index, cnt.values,
                 color=["#C44E52" if p < 10 else "#4C72B0" for p in pct])
axes[0].bar_label(b, labels=[f"{n}건 ({p:.1f}%)" for n, p in zip(cnt, pct)], padding=4, fontsize=9)
axes[0].set_xlim(0, cnt.max() * 1.3)
axes[0].set_xlabel("건수")
axes[0].set_title("결함 유형별 데이터 분포 (빨강: 10% 미만)")

pos = np.arange(len(order))
axes[1].barh(pos + 0.2, ratio.loc[order, "Train 비율"] * 100, height=0.4, color="#4C72B0", label="Train")
axes[1].barh(pos - 0.2, ratio.loc[order, "Test 비율"] * 100, height=0.4, color="#55A868", label="Test")
axes[1].set_yticks(pos, order)
axes[1].set_xlabel("비율 (%)")
axes[1].set_title("Train / Test 비율 비교 (층화 분할 확인)")
axes[1].legend(loc="lower right")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

**확인 결과**
- 분할 무결성 검사는 모두 0건으로 정상입니다.
- 유형별 비율 차이는 최대 0.0011로, 층화 분할이 정상적으로 이루어졌습니다.
- Test의 Stains(14건)·Dirtiness(11건)는 1건 차이로 Recall이 각각 0.071, 0.091씩 움직입니다. → **최종 Test 결과에서 이 두 유형의 수치는 변동 폭을 감안해 해석합니다.**

## 6. README 그림: 유형별 건수와 Baseline F1 (README 5.2)

README 5.2의 그림은 "건수가 적은 유형이 곧 어려운 유형인가?"를 보여 줍니다. 이를 위해 **03의 Baseline(XGBoost 기본값)과 같은 설정**으로 유형별 F1을 다시 계산합니다.

- Train 데이터만, 원본 변수 26개, 층화 5-fold (shuffle, `random_state=42`)
- 각 fold의 검증 예측을 모아(out-of-fold) 유형별 F1을 계산합니다. README 7.1의 Baseline 유형별 표와 같은 방식입니다.
- 그림 설명용 재계산이며, 이 결과로 모델이나 설정을 고르지 않습니다. Test는 사용하지 않습니다.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

X_tr = df_processed.loc[tr].drop(columns="target")
y_tr = df_processed.loc[tr, "target"]
classes = np.unique(y_tr)
y_tr_num = np.searchsorted(classes, y_tr)            # XGBoost용 숫자 라벨 (알파벳 순)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baseline = XGBClassifier(random_state=42, eval_metric="mlogloss")

fold_macro = [f1_score(y_tr_num[b], baseline.fit(X_tr.iloc[a], y_tr_num[a]).predict(X_tr.iloc[b]), average="macro")
              for a, b in cv.split(X_tr, y_tr_num)]
oof = cross_val_predict(baseline, X_tr, y_tr_num, cv=cv)
baseline_f1 = pd.Series(f1_score(y_tr_num, oof, average=None), index=classes)

print(f"Macro F1 (fold 평균 ± 표준편차): {np.mean(fold_macro):.4f} ± {np.std(fold_macro):.4f}   (README 7.1: 0.8109 ± 0.0308)")
baseline_f1.sort_values(ascending=False).round(3).to_frame("Baseline F1 (out-of-fold)")

In [ ]:
# README 5.2 그림 → figures/target_distribution.png
total = y_all.value_counts()                          # 전체 건수, 큰 순서
n_tr = y_all.loc[tr].value_counts()
n_te = y_all.loc[te].value_counts()
hard = baseline_f1 < 0.8                              # Baseline F1 0.8 미만 → 서로 헷갈리는 유형 (03 혼동행렬에서 확인)
DARK = {True: "#E07B2A", False: "#2E6FB7"}
LIGHT = {True: "#F2C9A8", False: "#AEC6E4"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.4))

# ① 유형별 건수 (Train + Test)
order = total.index[::-1]                             # 큰 값이 위로
ax1.barh(order, n_tr[order], color=[DARK[hard[c]] for c in order])
ax1.barh(order, n_te[order], left=n_tr[order], color=[LIGHT[hard[c]] for c in order])
for i, c in enumerate(order):
    ax1.text(total[c] + 8, i, f"{total[c]}건 ({total[c] / total.sum():.1%})", va="center", fontsize=10)
ax1.set_xlim(0, 800)
ax1.set_xlabel("결함 건수")
ax1.set_title("① 결함 유형별 건수 (진한 색 Train, 연한 색 Test)", fontweight="bold", pad=16)
ax1.text(1, 1.01, f"가장 많은 유형 ÷ 가장 적은 유형 = {total.max() / total.min():.1f}배",
         transform=ax1.transAxes, ha="right", va="bottom", fontsize=9.5, color="dimgray")

# ② 건수와 Baseline F1
ax2.axhspan(0.5, 0.8, color="#FBF3EC", zorder=0)
ax2.text(0.98, 0.04, "F1 0.8 미만 구간", transform=ax2.transAxes, ha="right", color=DARK[True], fontsize=9)
OFFSET = {"K_Scratch": (-15, 16, "right"), "Z_Scratch": (12, 0, "left"), "Stains": (12, -18, "left"),
          "Dirtiness": (12, -18, "left"), "Other_Faults": (-12, 18, "right"), "Bumps": (12, 0, "left"),
          "Pastry": (12, -4, "left")}
for c in classes:
    ax2.scatter(total[c], baseline_f1[c], s=120, color=DARK[hard[c]], edgecolor="white", zorder=3)
    dx, dy, ha = OFFSET[c]
    ax2.annotate(c, (total[c], baseline_f1[c]), xytext=(dx, dy), textcoords="offset points",
                 ha=ha, va="center", fontsize=10)
ax2.set_xlim(0, total.max() * 1.13)
ax2.set_ylim(0.5, 1.02)
ax2.set_xlabel("결함 건수")
ax2.set_ylabel("F1 (Baseline XGBoost, Train 5-fold CV)")
ax2.set_title("② 건수가 적다고 맞히기 어려운 것은 아니다", fontweight="bold", pad=16)
ax2.legend([plt.Rectangle((0, 0), 1, 1, color=DARK[False]), plt.Rectangle((0, 0), 1, 1, color=DARK[True])],
           ["잘 구분되는 유형", "서로 헷갈리는 유형"], loc="lower right", bbox_to_anchor=(1, 0.08), frameon=False)

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / "target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**확인 결과**
- 건수가 가장 적은 Stains(F1 0.939)·Dirtiness(0.854)는 F1 0.8 이상입니다.
- F1이 가장 낮은 것은 중간 규모의 Pastry(0.561)이고, 건수가 많은 Bumps(0.676)·Other_Faults(0.752)가 그다음입니다.

**판단**
- 이 데이터의 어려움은 "건수가 적은 유형"이 아니라 **서로 닮은 유형(Bumps·Pastry·Other_Faults)**에 있을 가능성이 높습니다. 세 유형이 실제로 서로 뒤바뀌는지는 03의 Baseline 혼동행렬로 확인합니다.
- 따라서 불균형 처리는 "적은 유형 늘리기"만으로 접근하지 않고, 이 세 유형의 성능 변화를 함께 봅니다.

## 7. 전처리 결정 기록 (README 6.2)

| 처리 항목 | 확인 결과 | 적용 방법 | 근거 |
|---|---|---|---|
| 정답 형식 | 원-핫 7개 열, 모든 행에서 정확히 하나만 1 | 단일 `target` 생성 후 원-핫 열 제거 | 다중 분류 문제. 정답 열이 입력에 남으면 누수 (2장) |
| 컬럼명 오타 | `K_Scatch` | `K_Scratch`로 수정 | 표기 통일 (2장) |
| 결측치 | 0건 | 처리 없음 | 1장 |
| 완전 중복 | 0건 | 처리 없음 | 1장 |
| 유사 중복 | 의심 1쌍(1775·1776, Test·Train), 판단 보류 1쌍(529·678) | 두 쌍 모두 제거하지 않음 | 확정할 방법이 없음. Test 한 건이 이미 본 결함일 수 있다는 점은 한계로 기록 (1장) |
| 중복 강종 변수 | A300과 A400의 합이 항상 1 | `TypeOfSteel_A400` 제거 | 같은 정보의 반복 (3장) |
| 극단값 | 크기 계열 우측 왜도, 면적 극단값의 82.6%가 K_Scratch | 그대로 사용 | K_Scratch를 알아보는 단서 (01 EDA 5장) |
| 범주형 인코딩 | 입력 변수가 모두 숫자 | 하지 않음 | 강종은 이미 0/1 |
| 스케일링 | 변수마다 값의 범위가 크게 다름 | 여기서는 하지 않음. 로지스틱 회귀에만 Pipeline 안에서 표준화 | 통계를 학습하는 처리는 Train fold 안에서만 |
| 서로 비슷한 변수 | 위치·크기 계열 상관이 매우 높음 | 그대로 둠 | 서로 다른 물리적 특성. 동일 정보(A300/A400)만 제거 (01 EDA 6장) |
| 분할 | 층화 80:20 (`random_state=42`) | 팀 공통 인덱스 사용 | 유형 비율 차이 최대 0.0011 (5장) |